# 入門版 — 前日の天気から翌日の天気を予測してみよう

A、B、C、D、E、F という 6 つの地点があります。毎日の天気を `1`(快晴)か `0`(快晴ではない)で記録します。

確かめたいこと(仮説):

> **F 地点の今日の天気は、A〜E 地点の昨日の天気から予測できるのではないか?**

このノートブックでは、機械学習で仮説を検証するときの基本の流れを、最小限の道具で体験します。

1. データを作る(練習用のダミーデータ)
2. 「昨日の天気」を説明変数にする
3. データを「学習用」と「テスト用」に分ける
4. モデルを学習させて、テスト用データで答え合わせする
5. 結果を読み解く

もっと本格的な手順(検証データ、5 モデル比較、信頼区間など)は、同じフォルダーの
`weather-hypothesis.ipynb`(フル版)で扱っています。

**最初のセルは日本語フォントの読み込みのため、実行に数十秒かかります。**上から順に実行してください。

In [ ]:
import piplite
await piplite.install("matplotlib-fontja==1.1.0")

import matplotlib_fontja
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.neural_network import MLPClassifier

matplotlib_fontja.japanize()
SEED = 42

## 1. ダミーデータを作る

練習用に、365 日分の天気データを乱数で作ります(シードを 42 に固定しているので、誰が実行しても同じデータになります)。

- A〜E 地点: それぞれ 50% の確率で快晴になる、とてもシンプルな天気
- F 地点: **昨日の A・B・C の快晴が多いほど晴れやすい**、というルールで生成します。
  さらに「A と C が両方快晴だと、特に晴れやすい」というおまけのルールも入れます

$$\text{score} = -2.5 + 1.3 \times (A_{昨日} + B_{昨日} + C_{昨日}) + 1.2 \times A_{昨日} \times C_{昨日}$$

score が大きいほど F は晴れやすくなります(シグモイド関数で確率に変換)。

ポイント: **D と E は F のルールに入っていません**。あとで、モデルがこのことを見抜けるかも確かめます。

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))


rng = np.random.default_rng(SEED)
n_days = 365

# A〜E: 毎日 50% で快晴
X = rng.binomial(1, 0.5, size=(n_days, 5))

# F: 昨日の A, B, C から決まる(初日だけは前日が無いので 50%)
F = np.zeros(n_days, dtype=int)
F[0] = rng.binomial(1, 0.5)
for t in range(1, n_days):
    a, b, c, d, e = X[t - 1]
    score = -2.5 + 1.3 * (a + b + c) + 1.2 * a * c
    F[t] = rng.binomial(1, sigmoid(score))

weather = pd.DataFrame(X, columns=["A", "B", "C", "D", "E"])
weather.insert(0, "day", np.arange(1, n_days + 1))
weather["F"] = F

print("F 地点の快晴率:", round(weather["F"].mean(), 3))
weather.head()

## 2. 「昨日の天気」を説明変数にする

予測に使ってよいのは**昨日**の A〜E だけです。`shift(1)` で 1 日ずらした列を作ります。
1 日目は「昨日」が存在しないので、その行は削除します。

(もし**今日**の A〜E で今日の F を予測してしまうと、問題設定が変わってしまいます。
時系列データでは「予測に使える情報はいつの時点のものか」をいつも意識しましょう。)

In [ ]:
features = ["A_lag1", "B_lag1", "C_lag1", "D_lag1", "E_lag1"]

df = weather.copy()
for site in ["A", "B", "C", "D", "E"]:
    df[f"{site}_lag1"] = df[site].shift(1)  # 昨日の値
df = df.dropna().reset_index(drop=True)

df[["day", "A_lag1", "B_lag1", "C_lag1", "D_lag1", "E_lag1", "F"]].head()

## 3. 学習用とテスト用に分ける

前から 70%(約 254 日)を**学習用**、残りの 30%(約 110 日)を**テスト用**にします。

大事なルールが 2 つあります。

- **時間の順番どおりに分ける**(シャッフルしない)。未来のデータで学習して過去を予測するのは反則だからです
- **テスト用データは最後の答え合わせまで一切使わない**

In [ ]:
n_train = int(len(df) * 0.7)
train, test = df.iloc[:n_train], df.iloc[n_train:]

X_train, y_train = train[features].to_numpy(), train["F"].to_numpy()
X_test, y_test = test[features].to_numpy(), test["F"].to_numpy()

print(f"学習用: {len(train)} 日 (day {train['day'].iloc[0]:.0f}〜{train['day'].iloc[-1]:.0f})")
print(f"テスト用: {len(test)} 日 (day {test['day'].iloc[0]:.0f}〜{test['day'].iloc[-1]:.0f})")

## 4. モデルを学習させて答え合わせする

3 つのモデルを比べます。

1. **多数派モデル** … いつも「多い方のクラス」を答えるだけのモデル。これが基準線です
2. **ロジスティック回帰** … 定番のシンプルなモデル
3. **ニューラルネットワーク** … 小さな隠れ層(8 ニューロン)を 1 つ持つモデル

成績は 2 つの指標で測ります。

- **Accuracy(正解率)** … 何割の日を当てられたか
- **ROC-AUC** … 予測の「確からしさの順位づけ」がどれだけ正しいか。0.5 なら当てずっぽうと同じ、1.0 なら完璧

In [ ]:
# 1. 多数派モデル: いつも学習データで多かった方を予測する
majority = int(y_train.mean() >= 0.5)
pred_majority = np.full(len(y_test), majority)

# 2. ロジスティック回帰
logreg = LogisticRegression(random_state=SEED).fit(X_train, y_train)
proba_lr = logreg.predict_proba(X_test)[:, 1]

# 3. ニューラルネットワーク
nn = MLPClassifier(hidden_layer_sizes=(8,), learning_rate_init=0.01,
                   max_iter=2000, random_state=SEED).fit(X_train, y_train)
proba_nn = nn.predict_proba(X_test)[:, 1]

results = pd.DataFrame({
    "Accuracy": [
        accuracy_score(y_test, pred_majority),
        accuracy_score(y_test, (proba_lr >= 0.5).astype(int)),
        accuracy_score(y_test, (proba_nn >= 0.5).astype(int)),
    ],
    "ROC-AUC": [
        0.5,
        roc_auc_score(y_test, proba_lr),
        roc_auc_score(y_test, proba_nn),
    ],
}, index=["多数派モデル", "ロジスティック回帰", "ニューラルネットワーク"])

results.round(3)

**読み方**: 多数派モデルの Accuracy が「何も考えずに達成できる点数」です。
ロジスティック回帰やニューラルネットワークがそれを上回り、AUC も 0.5 をはっきり超えていれば、
「昨日の A〜E には、今日の F を予測する情報が入っている」= 仮説を支持する結果です。

## 5. 予測の様子を目で見る

テスト期間について、実測の F(黒い点)と NN の予測確率(緑の線)を並べてみます。

In [ ]:
pred_nn = (proba_nn >= 0.5).astype(int)
days = test["day"].to_numpy()

plt.figure(figsize=(10, 3))
plt.plot(days, proba_nn, color="seagreen", label="NN の予測確率")
plt.scatter(days, y_test, color="black", s=15, zorder=3, label="実測 (0/1)")
plt.scatter(days[pred_nn != y_test], y_test[pred_nn != y_test],
            facecolors="none", edgecolors="crimson", s=80, zorder=4, label="外した日")
plt.axhline(0.5, color="gray", linestyle="--", alpha=0.6)
plt.title("テスト期間の F 地点 — 実測と予測")
plt.xlabel("day")
plt.legend(loc="center left", bbox_to_anchor=(1.01, 0.5))
plt.tight_layout()
plt.show()

print(f"テスト期間の的中: {int((pred_nn == y_test).sum())} / {len(y_test)} 日")

## 6. モデルはルールを見抜けたか?

### 昨日の A・B・C の 8 パターンで予測確率を見る

F のルールに関係するのは A・B・C の 3 地点でした。この 3 地点の組み合わせ($2^3 = 8$ 通り)ごとに、
NN の予測確率と、データを作るときに使った**本当の確率**を並べてみます。

In [ ]:
patterns = pd.DataFrame(
    [(a, b, c) for a in (0, 1) for b in (0, 1) for c in (0, 1)],
    columns=["A", "B", "C"],
)
patterns["本当の確率"] = sigmoid(
    -2.5 + 1.3 * (patterns["A"] + patterns["B"] + patterns["C"])
    + 1.2 * patterns["A"] * patterns["C"]
)

# NN には D, E も入力する必要があるので、D, E の 4 通りをならした平均を使う
nn_probs = []
for _, row in patterns.iterrows():
    combos = [[row["A"], row["B"], row["C"], d, e] for d in (0, 1) for e in (0, 1)]
    nn_probs.append(nn.predict_proba(np.array(combos))[:, 1].mean())
patterns["NNの予測確率"] = nn_probs

labels = patterns.apply(lambda r: f"A={r.A:.0f} B={r.B:.0f} C={r.C:.0f}", axis=1)
x = np.arange(8)
plt.figure(figsize=(9, 3.2))
plt.bar(x - 0.2, patterns["本当の確率"], width=0.4, label="本当の確率(生成ルール)")
plt.bar(x + 0.2, patterns["NNの予測確率"], width=0.4, label="NN の予測確率")
plt.xticks(x, labels, rotation=30)
plt.ylabel("F が快晴になる確率")
plt.title("昨日の A・B・C の 8 パターンと F の確率")
plt.legend()
plt.tight_layout()
plt.show()

2 色の棒がだいたい同じ高さなら、NN はデータを作った本当のルールをほぼ言い当てたことになります。
「A=1 B=0 C=1」が「A=1 B=1 C=0」より高ければ、おまけのルール(A と C がそろうと特に晴れる)まで
見抜けています。

### どの地点が大事か — シャッフルして壊してみる

ある地点の列だけを**バラバラに並べ替えて**(情報を壊して)、成績(ROC-AUC)がどれだけ下がるかを測ります。
下がりが大きい地点ほど、モデルの予測にとって大事な地点です。
ルール上、**D と E は F と無関係**なので、下がりはほぼゼロになるはずです。

In [ ]:
base_auc = roc_auc_score(y_test, proba_nn)
rng_shuffle = np.random.default_rng(SEED)

drops = []
for j, name in enumerate(features):
    aucs = []
    for _ in range(30):
        Xp = X_test.copy()
        rng_shuffle.shuffle(Xp[:, j])
        aucs.append(roc_auc_score(y_test, nn.predict_proba(Xp)[:, 1]))
    drops.append(base_auc - np.mean(aucs))

plt.figure(figsize=(6.5, 3))
plt.bar(["A", "B", "C", "D", "E"], drops, color="steelblue")
plt.axhline(0, color="gray", linewidth=0.8)
plt.title("その地点の情報を壊すと成績 (AUC) はどれだけ下がるか")
plt.ylabel("AUC の低下")
plt.tight_layout()
plt.show()

for name, d in zip(["A", "B", "C", "D", "E"], drops):
    print(f"{name} 地点: AUC の低下 {d:+.3f}")

## 7. まとめ

- **仮説は支持されました**: 昨日の A〜E を使ったモデルは、多数派モデルを上回り、AUC も 0.5 をはっきり超えました
- NN の予測確率は、データを作った**本当のルールとよく一致**しました(8 パターンの棒グラフ)
- シャッフル実験で、モデルが **A・C・B を重視し、無関係な D・E をほぼ無視**していることも確認できました

### ⚠️ 大事な注意

このデータは、答えのルールを知っている私たちが作った**練習用のダミーデータ**です。
うまく予測できたのは「分析の手順が正しく動いた」ということであって、
**現実の天気がこのルールで動いていることの証明ではありません**。

### 次のステップ

同じフォルダーの `weather-hypothesis.ipynb`(フル版)では、これに加えて

- 検証データを使った正しいモデル選び(ハイパーパラメータ調整)
- 決定木・ランダムフォレストも含めた 5 モデル比較と 6 つの評価指標
- 32 パターン全部の分析、Permutation Importance(AUC 版)、ブートストラップ信頼区間

を扱っています。この入門版が理解できたら、ぜひ挑戦してみてください。